# Research, part B: build the model

**This one starts from pre-fetched data on purpose.** No network, no waiting, nothing to go wrong before you get to the interesting part. Run it in a second Claude session alongside [`01a_research_data.ipynb`](01a_research_data.ipynb).

The goal: rebuild [ARGO](https://www.pnas.org/doi/10.1073/pnas.1515373112) (Yang, Santillana, Kou, *PNAS* 2015), which took me the better part of a year as a PhD student, in three prompts and no hand-written Python.

Run the cell below first. It is already filled in so you lose no time.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})

gt = pd.read_csv("../data/gt_dengue_mx.csv", parse_dates=["date"]).set_index("date")
od = pd.read_csv("../data/opendengue_mexico_monthly.csv", parse_dates=["month"]).set_index("month")

# 2015-2019 is the longest run of contiguous months on one case definition.
# Part A is where that gets discovered; here we take it as given.
LO, HI, WINDOW = "2015-01-01", "2019-12-01", 24
cases  = od["cases"].loc[LO:HI]
search = gt.loc[cases.index]
print(f"{len(cases)} months, {cases.index.min():%Y-%m} to {cases.index.max():%Y-%m}, "
      f"mean {cases.mean():,.0f} cases/month")


## Prompt 1: the honest baseline

Before anything clever, reproduce what a reasonable person would have done in 2011.

```
Plot cases against the "dengue" search column over time on a twin axis, and print the
correlation of every search column with cases. Then fit ordinary least squares of cases
on the single "dengue" column, training only on the first 24 months. Predict the rest
from that frozen fit. Do not refit on anything later.
```

That last sentence is load-bearing. Leakage is the most common way this goes quietly wrong, and an agent optimizing for a good-looking number will refit if you let it.

In [ ]:
# Prompt 1: paste the agent's code here and run it.


## Prompt 2: now make it ARGO

```
Now work in log space. Regress log cases on the logs of all four search terms plus
three autoregressive lags of log cases. Use L1 regularization with a cross-validated
penalty, and retrain on a rolling 24-month window at every step so the model only ever
sees the past. Fit two references on the identical rolling scheme: autoregression only,
and search only. Return predictions on the original scale.
```

Two structural ideas carry ARGO and both are in that paragraph: **autoregression** (yesterday's dengue predicts today's) and **dynamic training** (the search-to-disease relationship drifts, so keep refitting).

In [ ]:
# Prompt 2: paste the agent's code here and run it.


## Stop. What did it decide for you?

Ask before you look at any result:

```
Walk me through what you just did, line by line. Where did you have to make a choice
I did not specify? What would break if my data were slightly different?
```

Then verify the answer instead of taking it. If it says anything about the search columns, go and count:

In [ ]:
# How many search values are exactly zero, and what share of the record is that?


Hold two things at once. The patch it made was probably **correct**. It was also probably **silent**, and a silent correct patch and a silent wrong one look identical from the outside.

Whatever you found is not a numerical nuisance, it is a *measurement* fact about how the data source reports low volume. That changes what the model can honestly claim, and no prompt produces that sentence. It comes from knowing the source.

> The agent closes the gap between having an idea and seeing a number, almost completely. It does not close the gap between seeing a number and believing it. That gap is still the job.

## Prompt 3: score everything against each other

```
Build one comparison table over the common evaluation window: RMSE, MAE and correlation
for the static baseline, the autoregression-only model, the search-only model, and ARGO.
Add a column giving each model's RMSE relative to the autoregression benchmark. Then plot
ARGO against the truth over time.
```

In [ ]:
# Prompt 3: paste the agent's code here and run it.


## Now read the table honestly

The agent will not do this part, and it is the part that decides whether the analysis is any good. Answer these yourself before looking at mine:

* Which single change bought the most accuracy?
* How much did search add **over and above** the disease's own history?
* How does search do **on its own**? Sufficient, or only complementary?
* How many evaluation points is this margin based on? Enough to claim anything?

**Then compare with [`01_research_soln.ipynb`](01_research_soln.ipynb)**, which has every output executed and my reading of them.

---

**Next:** [`02_education_reading_group.md`](../02_education_reading_group.md), where the agent is not writing models at all.